# 05 — Hybrid Retrieval and Reranking

**Purpose:** Combine dense and BM25 retrieval with reciprocal-rank fusion, then rerank the final candidates.

Retrieval logic lives in `laborlaw_rag.search`; this notebook makes one embedding request and, when enabled, one reranking request.


## 1. Setup


In [ ]:
from laborlaw_rag.config import RAGConfig, Settings
from laborlaw_rag.data import load_chunks
from laborlaw_rag.search import HybridRetriever, normalize_persian
from laborlaw_rag.services import JinaClient

settings = Settings.from_env()
config = RAGConfig.from_env()
QUESTION = "شرایط پرداخت حق سنوات به کارگر چیست؟"

## 2. Load Retrieval Resources


In [ ]:
chunks = load_chunks(settings.chunks_path, settings.source_url)
retriever = HybridRetriever.from_artifacts(
    chunks,
    JinaClient(settings),
    settings,
    config,
)

## 3. Retrieve and Rerank


In [ ]:
normalized_query = normalize_persian(QUESTION)
hits = retriever.retrieve([normalized_query], normalized_query)

## 4. Inspect Ranked Evidence


In [ ]:
[
    {
        "rank": rank,
        "article_reference": hit.chunk.article_reference,
        "fusion_score": round(hit.fusion_score, 6),
        "rerank_score": (round(hit.rerank_score, 6) if hit.rerank_score is not None else None),
        "text_preview": hit.chunk.text[:240],
    }
    for rank, hit in enumerate(hits, start=1)
]